1. Define the list of cities.
2. Retrieve GPS coordinates using the Nominatim API.

In [ ]:
from pathlib import Path
import pandas as pd
import requests
import time

# Project folders
PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#Add the 35 cities
cities = [
    "Mont Saint Michel",
    "St Malo",
    "Bayeux",
    "Le Havre",
    "Rouen",
    "Paris",
    "Amiens",
    "Lille",
    "Strasbourg",
    "Chateau du Haut Koenigsbourg",
    "Colmar",
    "Eguisheim",
    "Besancon",
    "Dijon",
    "Annecy",
    "Grenoble",
    "Lyon",
    "Gorges du Verdon",
    "Bormes les Mimosas",
    "Cassis",
    "Marseille",
    "Aix en Provence",
    "Avignon",
    "Uzes",
    "Nimes",
    "Aigues Mortes",
    "Saintes Maries de la mer",
    "Collioure",
    "Carcassonne",
    "Ariege",
    "Toulouse",
    "Montauban",
    "Biarritz",
    "Bayonne",
    "La Rochelle"
]

len(cities)

In [ ]:
#Testing one city
url = "https://nominatim.openstreetmap.org/search"

params = {
    "city": "Paris",
    "country": "France",
    "format": "json",
    "limit": 1
}

headers = {
    "User-Agent": "jedha-kayak-project"
}

response = requests.get(
    url,
    params=params,
    headers=headers
)

response.status_code

In [ ]:
# Inspect what Nominatim returned
response.json()

In [ ]:
# First API extraction
data = response.json()

latitude = data[0]["lat"]
longitude = data[0]["lon"]

print(latitude, longitude)

In [ ]:
# Reusable function
def get_coordinates(city):

    url = "https://nominatim.openstreetmap.org/search"

    params = {
        "q": f"{city}, France",
        "format": "json",
        "limit": 1
    }

    headers = {
        "User-Agent": "jedha-kayak-project"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    response.raise_for_status()

    data = response.json()

    if data:
        return {
            "city": city,
            "latitude": float(data[0]["lat"]),
            "longitude": float(data[0]["lon"])
        }

    return {
        "city": city,
        "latitude": None,
        "longitude": None
    }

In [ ]:
# Test Paris
get_coordinates("Paris")

In [ ]:
# Get coordinates for all 35 destinations
coordinates = []

for city in cities:

    print(f"Getting coordinates for {city}...")

    result = get_coordinates(city)
    coordinates.append(result)

    time.sleep(1)

In [ ]:
# Create DataFrame

cities_df = pd.DataFrame(coordinates)

cities_df.head()

In [ ]:
# Check Shape
cities_df.shape

In [ ]:
# Missing values
cities_df.isna().sum()

In [ ]:
# Create a city_id column
cities_df.insert(0, "city_id", range(1, len(cities_df) + 1))
cities_df.head()

In [ ]:
cities_df["city_id"].is_unique

In [ ]:
# Save the coordinates dataset
cities_df.to_csv(RAW_DATA_DIR / "cities_coordinates.csv", index=False)